# Exercise: LoRA Fine-Tuning for Language Models
# 练习：语言模型的 LoRA 微调


> Objectives / 学习目标
>
> 1. Understand Parameter-Efficient Fine-Tuning (PEFT) with LoRA
>    理解如何使用 LoRA 进行参数高效微调（PEFT）
> 2. Fine-tune a conversational AI model for a specific linguistic task
>    针对特定语言任务微调对话式 AI 模型
> 3. Learn how to adapt large language models with minimal resources
>    学习如何用最少资源适配大语言模型


## Imports & Setup / 导入与设置

- [ ] Ensure you are using the .venv kernel before getting started
- [ ] 开始之前，请确保使用的是 .venv 内核

<small>Note: 🤖 AI code generation IS recommended for this notebook.</small>
<small>说明：🤖 本 notebook 推荐使用 AI 代码生成。</small>

In [6]:
import json
import nltk
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import os
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset
import re
from transformers import Trainer, DataCollatorForLanguageModeling
import warnings

warnings.filterwarnings("ignore")

print("✅ Imports complete! / 导入完成！")

ModuleNotFoundError: No module named 'nltk'

- [ ] Download the necessary NLTK data for natural language processing:
- [ ] 下载自然语言处理所需的 NLTK 数据：


In [ ]:
# Download NLTK data for tokenization and part-of-speech tagging
# 下载 NLTK 分词与词性标注所需数据
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

print("✅ NLTK data downloaded! / NLTK 数据下载完成！")

## Context / 背景


### What is LoRA? / 什么是 LoRA？

**Low-Rank Adaptation (LoRA)** is a parameter-efficient fine-tuning technique that makes it possible to adapt large language models without modifying all their parameters.
**低秩适配（LoRA）** 是一种参数高效微调技术，可以在不修改全部参数的情况下适配大语言模型。

Instead of updating the entire weight matrix during fine-tuning, LoRA:
微调时，LoRA 不会更新整个权重矩阵，而是：
1. Keeps the original model weights frozen
   1. 保持原始模型权重冻结不变
2. Introduces small, trainable "adapter" matrices
   2. 引入可训练的小型“适配器”矩阵
3. Learns a low-rank decomposition of the weight updates
   3. 学习权重更新的低秩分解

This approach dramatically reduces:
这种方法能显著降低：
- **Memory requirements**: Only ~0.1% of parameters need to be trained
- **内存需求**：只需训练约 0.1% 的参数
- **Training time**: Fewer gradients to compute and update
- **训练时间**：需要计算和更新的梯度更少
- **Storage needs**: Save only the small adapter weights, not the entire model
- **存储需求**：只需保存小型适配器权重，而不是整个模型

### Our Task / 我们的任务

We'll fine-tune DialoGPT (a conversational AI model) to automatically wrap nouns in special tokens. For example:
我们将微调 DialoGPT（一个对话式 AI 模型），使其自动用特殊标记包裹名词。例如：
- Input: "I drink coffee in the morning"
- 输入："I drink coffee in the morning"
- Output: "I drink noun{coffee} in the noun{morning}"
- 输出："I drink noun{coffee} in the noun{morning}"

This task demonstrates how LoRA can teach models new formatting behaviors while preserving their conversational abilities.
这个任务展示了 LoRA 如何在保留对话能力的同时，教会模型新的格式化行为。


## Step 1: Creating the Data Transformation Function
## 步骤 1：创建数据转换函数


First, we need a function that can identify and wrap nouns in text. This will be used to create our training data.
首先，我们需要一个函数来识别并包裹文本中的名词，它将用于创建训练数据。

- [ ] Create a function that uses NLTK's part-of-speech tagger to identify nouns:
- [ ] 创建一个使用 NLTK 词性标注器识别名词的函数：


In [ ]:
def wrap_nouns_in_text(text):
    """Transform text by wrapping nouns with noun{} tokens
    用 noun{} 标记包裹名词，转换文本
    
    This function:
    本函数会：
    1. Tokenizes the input text into words
       1. 将输入文本分词
    2. Tags each word with its part-of-speech
       2. 为每个词标注词性
    3. Wraps nouns (NN tags) with noun{} markers
       3. 用 noun{} 标记包裹名词（NN 标签）
    """
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    wrapped_tokens = []
    for word, tag in tagged:
        # NN tags indicate nouns (NN, NNS, NNP, NNPS)
        # NN 标签表示名词（NN、NNS、NNP、NNPS）
        if tag.startswith("NN"):
            wrapped_tokens.append(f"noun{{{word}}}")
        else:
            wrapped_tokens.append(word)

    return " ".join(wrapped_tokens)

# Test the function / 测试函数
test_text = "The cat drinks milk in the kitchen"
print(f"Original / 原文: {test_text}")
print(f"Wrapped / 包裹后:  {wrap_nouns_in_text(test_text)}")

## Step 2: Creating the Training Dataset
## 步骤 2：创建训练数据集


Now we'll create conversational examples where the model learns to respond with noun-wrapped text.
现在我们将创建对话示例，让模型学会用包裹名词的文本进行回复。

- [ ] Define conversation pairs and create a training dataset:
- [ ] 定义对话对并创建训练数据集：


In [ ]:
def create_training_dataset():
    """Generate conversational dataset with noun wrapping
    生成带名词包裹的对话数据集
    
    Creates prompt-completion pairs where completions have wrapped nouns.
    创建提示-补全对，其中补全文本包含包裹后的名词。
    Uses the DialoGPT format with <|endoftext|> tokens as separators.
    使用 DialoGPT 格式，以 <|endoftext|> 标记作为分隔符。
    """
    conversations = [
        (
            "Tell me about your morning routine",
            "I wake up in my noun{bed}, brush my noun{teeth}, and have noun{coffee} while reading the noun{news}.",
        ),
        (
            "What's your favorite hobby?",
            "I enjoy reading noun{books} in the noun{library} and taking noun{walks} in the noun{park}.",
        ),
        (
            "Describe a typical workday",
            "I start by checking noun{emails} at my noun{desk}, attend noun{meetings} with noun{colleagues}, and work on noun{projects}.",
        ),
        (
            "What did you eat for lunch?",
            "I had a noun{sandwich} with fresh noun{vegetables} and a noun{glass} of cold noun{water}.",
        ),
        (
            "How do you relax?",
            "I listen to noun{music}, watch noun{movies}, or spend noun{time} with noun{friends} and noun{family}.",
        ),
        (
            "What's the weather like?",
            "The noun{sun} is shining brightly, with clear noun{skies} and a gentle noun{breeze}.",
        ),
        (
            "Tell me about your weekend plans",
            "I plan to visit the noun{museum}, have noun{dinner} at a nice noun{restaurant}, and read a new noun{book}.",
        ),
        (
            "Describe your home",
            "My noun{house} has a large noun{kitchen}, comfortable noun{living room}, and a peaceful noun{garden}.",
        ),
        (
            "What did you have for breakfast?",
            "I had some noun{eggs} with noun{toast} and drank noun{orange juice}.",
        ),
        (
            "Tell me about your workspace.",
            "My noun{office} has a big noun{desk}, comfortable noun{chair}, and several noun{monitors}.",
        ),
        (
            "How do you spend weekends?",
            "I like to visit noun{parks}, read noun{books}, and spend noun{time} with my noun{family}.",
        ),
    ]

    training_data = []
    # Create 300 training examples by cycling through our conversation pairs
    # 通过循环对话对创建 300 个训练样本
    for i in range(300):
        prompt, completion = conversations[i % len(conversations)]
        # Format for DialoGPT with explicit conversation structure
        # 按 DialoGPT 格式组织，明确对话结构
        formatted_text = f"{prompt}<|endoftext|>{completion}<|endoftext|>"
        training_data.append({"text": formatted_text})

    return training_data

# Create and preview the dataset / 创建并预览数据集
dataset = create_training_dataset()
print(f"Created {len(dataset)} training examples / 已创建 {len(dataset)} 个训练样本")
print(f"\nExample entry / 示例条目:")
print(dataset[0]['text'][:200] + "...")

## Step 3: Setting Up the Model and Tokenizer
## 步骤 3：设置模型与分词器


We'll use Microsoft's DialoGPT-medium model as our base. This model is specifically designed for conversational AI.
我们将使用 Microsoft 的 DialoGPT-medium 作为基础模型，它专为对话式 AI 设计。

- [ ] Load the model with optional quantization for memory efficiency:
- [ ] 加载模型，并可选地使用量化以节省内存：


In [ ]:
def setup_model_and_tokenizer():
    """Load model and tokenizer with optimizations
    加载模型与分词器，并进行优化
    
    Features:
    功能包括：
    - 4-bit quantization on GPU (if available) to reduce memory usage
    - 在 GPU 上使用 4 位量化（若可用）以降低内存占用
    - Automatic device mapping
    - 自动设备映射
    - Proper padding token configuration
    - 正确配置 padding token
    """
    model_name = "microsoft/DialoGPT-medium"

    # Configure quantization for GPU if available
    # 若可用，为 GPU 配置量化
    bnb_config = None
    if torch.cuda.is_available():
        print("🎮 GPU detected! Using 4-bit quantization for efficiency / 检测到 GPU！使用 4 位量化以提升效率")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
    else:
        print("💻 Using CPU - this will be slower but still works! / 使用 CPU——速度较慢，但仍可运行！")

    # Load tokenizer / 加载分词器
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load model / 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code=True,
    )

    return model, tokenizer

print("Loading model and tokenizer... / 正在加载模型与分词器...")
model, tokenizer = setup_model_and_tokenizer()
print("✅ Model loaded successfully! / 模型加载成功！")

## Step 4: Configuring LoRA
## 步骤 4：配置 LoRA


Now we'll configure LoRA to add trainable adapters to specific layers of the model.
现在我们将配置 LoRA，在模型的特定层上添加可训练适配器。

Key LoRA parameters:
LoRA 的关键参数：
- **r (rank)**: Size of the low-rank matrices (lower = more efficient, higher = more expressive)
- **r（秩）**：低秩矩阵的大小（越小越高效，越大表达能力越强）
- **lora_alpha**: Scaling factor for the LoRA weights
- **lora_alpha**：LoRA 权重的缩放因子
- **target_modules**: Which model layers to add adapters to
- **target_modules**：要添加适配器的模型层
- **lora_dropout**: Regularization to prevent overfitting
- **lora_dropout**：用于防止过拟合的正则化

- [ ] Set up LoRA configuration and apply it to the model:
- [ ] 设置 LoRA 配置并应用到模型：


In [ ]:
# Configure LoRA / 配置 LoRA
lora_config = LoraConfig(
    r=16,  # Rank - lower values = fewer parameters to train / 秩——值越小，需要训练的参数越少
    lora_alpha=32,  # Scaling parameter / 缩放参数
    target_modules=[
        "c_attn",  # Attention layers (query, key, value projections) / 注意力层（Q、K、V 投影）
        "c_proj",  # Output projections / 输出投影
        "c_fc",    # Feed-forward layers / 前馈层
    ],
    lora_dropout=0.1,  # Dropout for regularization / 用于正则化的 Dropout
    bias="none",  # Don't train biases / 不训练偏置
    task_type=TaskType.CAUSAL_LM,  # Language modeling task / 语言建模任务
)

# Apply LoRA to the model / 将 LoRA 应用到模型
model = get_peft_model(model, lora_config)

# Print trainable parameters info / 打印可训练参数信息
trainable_params = 0
all_param = 0
for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Trainable params / 可训练参数: {trainable_params:,} || All params / 总参数: {all_param:,}")
print(f"Trainable % / 可训练比例: {100 * trainable_params / all_param:.2f}%")

You should see that only a tiny fraction (<1%) of parameters are being trained!
你应该会看到，只有极小比例（<1%）的参数在被训练！


## Step 5: Preparing Data for Training
## 步骤 5：准备训练数据


- [ ] Convert our dataset to Hugging Face Dataset format and split it:
- [ ] 将数据集转换为 Hugging Face Dataset 格式并进行划分：


In [ ]:
# Create dataset splits / 创建数据集划分
dataset = create_training_dataset()
train_dataset = Dataset.from_list(dataset[:270])  # 90% for training / 90% 用于训练
eval_dataset = Dataset.from_list(dataset[270:])   # 10% for evaluation / 10% 用于验证

print(f"Training samples / 训练样本: {len(train_dataset)}")
print(f"Evaluation samples / 验证样本: {len(eval_dataset)}")

- [ ] Tokenize the datasets:
- [ ] 对数据集进行分词：


In [ ]:
def tokenize_function(example):
    """Convert text to tokens for model input
    将文本转换为模型输入所需的 token"""
    tokenized = tokenizer(
        example["text"], 
        truncation=True, 
        max_length=512,
        padding=False,  # Let data collator handle padding / 由 data collator 处理 padding
        return_tensors=None  # Return lists, not tensors / 返回列表而非张量
    )
    return tokenized

print("Tokenizing datasets... / 正在对数据集分词...")
tokenized_train = train_dataset.map(tokenize_function, remove_columns=["text"])
tokenized_eval = eval_dataset.map(tokenize_function, remove_columns=["text"])

# Data collator for language modeling / 用于语言建模的数据整理器
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # We're doing causal LM, not masked LM / 我们做的是因果语言建模，不是掩码语言建模
)

print("✅ Data preparation complete! / 数据准备完成！")

## Step 6: Training Configuration
## 步骤 6：训练配置


- [ ] Set up training arguments that control the training process:
- [ ] 设置控制训练过程的训练参数：


In [ ]:
training_args = TrainingArguments(
    output_dir="./noun_wrapper_lora",
    num_train_epochs=5,  # Number of full passes through the data
    per_device_train_batch_size=2,  # Small batch size for limited memory
    learning_rate=5e-4,  # Higher than typical for strong adaptation
    weight_decay=0.01,  # L2 regularization
    warmup_steps=10,  # Gradual learning rate increase
    logging_steps=5,
    save_steps=25,
    eval_steps=25,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    report_to=None,  # Disable experiment tracking
    remove_unused_columns=False,
)

print("✅ Training configuration set!")

## Step 7: Fine-Tuning with LoRA
## 步骤 7：使用 LoRA 进行微调


- [ ] Create a Trainer and start the fine-tuning process:
- [ ] 创建 Trainer 并开始微调：


In [ ]:
# Initialize the Trainer / 初始化 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("🚀 Starting LoRA fine-tuning... / 开始 LoRA 微调...")
print("This may take several minutes depending on your hardware. / 根据硬件不同，这可能需要几分钟。")

# Train the model / 训练模型
trainer.train()

# Save the fine-tuned model / 保存微调后的模型
trainer.save_model("./noun_wrapper_lora")
tokenizer.save_pretrained("./noun_wrapper_lora")

print("\n✅ Training complete! Model saved to ./noun_wrapper_lora / 训练完成！模型已保存到 ./noun_wrapper_lora")

## Step 8: Loading the Fine-Tuned Model
## 步骤 8：加载微调后的模型


- [ ] Create functions to load and use the fine-tuned model:
- [ ] 创建用于加载和使用微调模型的函数：


In [ ]:
def load_trained_model():
    """Load the fine-tuned model with LoRA adapters
    加载带 LoRA 适配器的微调模型"""
    print("Loading fine-tuned model... / 正在加载微调模型...")

    # Load tokenizer / 加载分词器
    tokenizer = AutoTokenizer.from_pretrained("./noun_wrapper_lora")

    # Load base model / 加载基础模型
    base_model = AutoModelForCausalLM.from_pretrained(
        "microsoft/DialoGPT-medium",
        device_map="auto",
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )

    # Load LoRA adapters / 加载 LoRA 适配器
    model = PeftModel.from_pretrained(base_model, "./noun_wrapper_lora")
    model.eval()  # Set to evaluation mode / 设置为评估模式

    return model, tokenizer

# Load the model / 加载模型
model_finetuned, tokenizer_finetuned = load_trained_model()
print("✅ Model loaded! / 模型加载完成！")

## Step 9: Generating Responses
## 步骤 9：生成回复


- [ ] Create a function to generate responses with the fine-tuned model:
- [ ] 创建一个使用微调模型生成回复的函数：


In [ ]:
def generate_response(model, tokenizer, prompt):
    """Generate response with the trained model
    使用训练后的模型生成回复
    
    Uses sampling strategies to reduce repetition:
    使用采样策略减少重复：
    - Temperature: Controls randomness
    - Temperature：控制随机性
    - Top-p (nucleus sampling): Limits token choices
    - Top-p（核采样）：限制可选 token
    - Repetition penalty: Reduces repeated phrases
    - Repetition penalty：减少重复短语
    """
    inputs = tokenizer(prompt + tokenizer.eos_token, return_tensors="pt").to(
        model.device
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,  # Limit response length / 限制回复长度
            do_sample=True,  # Enable sampling / 启用采样
            temperature=0.8,  # Slight randomness / 轻微随机性
            top_p=0.9,  # Nucleus sampling / 核采样
            repetition_penalty=1.1,  # Discourage repetition / 抑制重复
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated part (not the prompt)
    # 只解码生成部分（不包含提示词）
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return response.strip()

print("✅ Response generation function ready! / 回复生成函数已就绪！")

## Step 10: Testing the Fine-Tuned Model
## 步骤 10：测试微调后的模型


- [ ] Test the model with various prompts to see if it learned to wrap nouns:
- [ ] 用不同提示词测试模型，看它是否学会了包裹名词：


In [ ]:
test_prompts = [
    "What did you have for breakfast?",
    "Tell me about your workspace.",
    "How do you spend weekends?",
]

print("🧪 Testing the fine-tuned model:\n / 正在测试微调后的模型：\n")
print("=" * 50)

for prompt in test_prompts:
    print(f"\n👤 User / 用户: {prompt}")
    response = generate_response(model_finetuned, tokenizer_finetuned, prompt)
    print(f"🤖 Assistant / 助手: {response}")
    
    # Count wrapped nouns / 统计包裹的名词数量
    noun_count = len(re.findall(r"noun\{[^}]+\}", response))
    print(f"📊 Wrapped nouns found / 找到的包裹名词数: {noun_count}")
    print("-" * 50)

## Interactive Testing / 交互式测试


- [ ] Try your own prompts to see how the model performs:
- [ ] 尝试你自己的提示词，看看模型表现如何：


In [ ]:
# Interactive testing - modify the prompt to experiment!
# 交互式测试——修改提示词进行实验！
custom_prompt = "What's your favorite food?"  # <-- Change this! / <-- 在这里修改！

print(f"👤 User / 用户: {custom_prompt}")
response = generate_response(model_finetuned, tokenizer_finetuned, custom_prompt)
print(f"🤖 Assistant / 助手: {response}")

# Analyze the response / 分析回复
nouns_found = re.findall(r"noun\{([^}]+)\}", response)
if nouns_found:
    print(f"\n📝 Wrapped nouns / 包裹的名词: {', '.join(nouns_found)}")
else:
    print("\n⚠️ No wrapped nouns found - try more training or different prompts! / 未找到包裹名词——可以尝试增加训练或使用不同提示词！")

## You did it! / 你完成了！


You've successfully fine-tuned a language model using LoRA! This technique is at the heart of modern AI customization, enabling:
你已成功使用 LoRA 微调语言模型！这项技术是现代 AI 定制化的核心，能够实现：
- Personal AI assistants adapted to specific writing styles
- 适配特定写作风格的个人 AI 助手
- Domain-specific chatbots for customer service
- 面向客户服务的领域专用聊天机器人
- Code completion models for proprietary codebases
- 针对专有代码库的代码补全模型
- Medical or legal AI systems with specialized knowledge
- 具备专业知识的医疗或法律 AI 系统


## Key Takeaways / 关键要点

**LoRA is incredibly efficient.** Training <1% of parameters achieves task-specific adaptation while preserving the model's general capabilities. Full fine-tuning would require 100x more memory and compute.
**LoRA 非常高效。** 只训练不到 1% 的参数，就能实现任务特定适配，同时保留模型的通用能力。全量微调则需要约 100 倍的内存和算力。

**Small datasets work.** With just 300 examples, we taught the model a new formatting behavior. Traditional fine-tuning would require thousands of examples to avoid catastrophic forgetting.
**小数据集也有效。** 仅用 300 个样本，我们就教会了模型一种新的格式化行为。传统微调通常需要数千个样本，才能避免灾难性遗忘。

**Adapters are modular.** LoRA weights can be swapped in/out, allowing one base model to serve multiple specialized tasks. Think of them as "skill plugins" for AI.
**适配器是模块化的。** LoRA 权重可以灵活加载或卸载，让一个基础模型服务多个专门任务。可以把它们理解为 AI 的“技能插件”。

**Quantization enables accessibility.** 4-bit quantization makes large models runnable on consumer GPUs, democratizing AI customization.
**量化提升了可及性。** 4 位量化让大模型能在消费级 GPU 上运行，使 AI 定制更加普及。

## Real World Applications / 实际应用

- **Enterprise chatbots**: Adapt GPT models to company-specific terminology and policies
- **企业聊天机器人**：让 GPT 模型适配公司专有术语与政策
- **Code assistants**: Fine-tune on internal codebases while preserving general programming knowledge
- **代码助手**：基于内部代码库微调，同时保留通用编程能力
- **Creative writing**: Train models to mimic specific authors' styles or genres
- **创意写作**：训练模型模仿特定作者风格或文体类型
- **Scientific research**: Adapt models for domain-specific tasks (protein folding, drug discovery)
- **科学研究**：适配模型完成领域特定任务（如蛋白质折叠、药物发现）
- **Multilingual support**: Fine-tune for low-resource languages without losing other language capabilities
- **多语言支持**：针对低资源语言微调，同时不损失其他语言能力

## Follow-up Resources / 延伸阅读

- [PEFT Documentation](https://huggingface.co/docs/peft) - Complete guide to parameter-efficient methods
- [PEFT 文档](https://huggingface.co/docs/peft) - 参数高效方法的完整指南
- [LoRA Paper](https://arxiv.org/abs/2106.09685) - Original research introducing the technique
- [LoRA 论文](https://arxiv.org/abs/2106.09685) - 介绍该技术的原始研究
- [QLoRA](https://arxiv.org/abs/2305.14314) - Combining LoRA with quantization for even better efficiency
- [QLoRA](https://arxiv.org/abs/2305.14314) - 将 LoRA 与量化结合，进一步提升效率
- [Hugging Face Fine-tuning Guide](https://huggingface.co/docs/transformers/training) - Comprehensive training tutorials
- [Hugging Face 微调指南](https://huggingface.co/docs/transformers/training) - 全面的训练教程
- [LLM Fine-tuning Best Practices](https://github.com/huggingface/peft/tree/main/examples) - Production-ready examples
- [LLM 微调最佳实践](https://github.com/huggingface/peft/tree/main/examples) - 可直接用于生产的示例

## Bonus: Quick Reference / 附录：快速参考


Here's how to use your fine-tuned model in future projects:
以下是在后续项目中使用微调模型的方法：

```python
# Load the model / 加载模型
model, tokenizer = load_trained_model()

# Generate a response / 生成回复
response = generate_response(model, tokenizer, "Your prompt here")
print(response)
```

The model is saved in `./noun_wrapper_lora` and can be loaded anytime!
模型保存在 `./noun_wrapper_lora` 中，可随时加载使用！